# ClearBank — Analise Financeira com Python

Notebook do desafio final. Le `transacoes.csv`, valida e limpa os dados,
calcula metricas mensais, sinaliza transacoes suspeitas, exibe um relatorio
no terminal e salva o resultado em `relatorio.json`.

## Sobre o arquivo de entrada (`transacoes.csv`)

Colunas: `id, data, cliente_id, tipo, valor, descricao, categoria`.
O arquivo de teste possui **17 registros validos** distribuidos em 4 meses,
**5 registros invalidos** (um para cada regra de validacao) e **2 transacoes
acima de R$ 10.000,00**.

> Crie/edite o `transacoes.csv` na mesma pasta do notebook antes de executar.

In [1]:
import csv
import json
from datetime import datetime

# ----- Constantes -----
ARQUIVO_CSV = "transacoes.csv"
ARQUIVO_JSON = "relatorio.json"
LIMITE_SUSPEITO = 10000.00
FORMATO_DATA = "%Y-%m-%d"
TIPOS_VALIDOS = ("credito", "debito")

## Formatacao monetaria (padrao brasileiro)

In [2]:
def formatar_brl(valor):
    """Formata um numero no padrao monetario brasileiro: R$ 1.234,56."""
    return f"R$ {valor:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")


# teste rapido
print(formatar_brl(1840.25))
print(formatar_brl(15000))

R$ 1.840,25
R$ 15.000,00


## Validacoes especificas

Duas funcoes pequenas e reutilizaveis — uma para a data, outra para o valor.
Ambas usam `try/except` quando chamadas dentro de `validar_transacao`.

In [3]:
def validar_data(texto):
    """Converte 'AAAA-MM-DD' em datetime. Lanca ValueError se invalida."""
    return datetime.strptime(texto.strip(), FORMATO_DATA)


def validar_valor(texto):
    """Converte o texto em float positivo. Lanca ValueError se invalido."""
    valor = float(texto)
    if valor <= 0:
        raise ValueError("valor deve ser maior que zero")
    return valor


# testes rapidos
print(validar_data("2026-01-05"))
print(validar_valor("180.50"))

2026-01-05 00:00:00
180.5


## Leitura do CSV

Usa `csv.DictReader` (modulo nativo, sem pandas) e trata `FileNotFoundError`
— **try/except #1**.

In [4]:
def ler_transacoes(caminho):
    """Le o CSV com csv.DictReader e retorna a lista de transacoes brutas."""
    try:
        with open(caminho, "r", encoding="utf-8") as arquivo:
            return list(csv.DictReader(arquivo))
    except FileNotFoundError:
        print(f"ERRO: arquivo '{caminho}' nao encontrado.")
        return []


print(f"Linhas lidas: {len(ler_transacoes(ARQUIVO_CSV))}")

Linhas lidas: 22


## Validacao e limpeza

`validar_transacao` aplica todas as regras a uma unica linha e devolve o
registro limpo ou `None`. Linhas invalidas sao descartadas silenciosamente.
Aqui ficam o **try/except #2 (data)** e o **try/except #3 (valor)**.

In [5]:
def validar_transacao(linha):
    """Valida uma linha bruta. Retorna o registro limpo ou None se invalida."""
    # id vazio ou nao numerico
    id_bruto = (linha.get("id") or "").strip()
    if not id_bruto.isdigit():
        return None

    # cliente_id vazio
    cliente_id = (linha.get("cliente_id") or "").strip()
    if not cliente_id:
        return None

    # tipo diferente de credito/debito
    tipo = (linha.get("tipo") or "").strip().lower()
    if tipo not in TIPOS_VALIDOS:
        return None

    # data invalida (try/except #2)
    try:
        data = validar_data(linha.get("data", ""))
    except ValueError:
        return None

    # valor nao numerico ou <= 0 (try/except #3)
    try:
        valor = validar_valor(linha.get("valor", ""))
    except (ValueError, TypeError):
        return None

    return {
        "id": int(id_bruto),
        "data": data,
        "mes": data.strftime("%Y-%m"),
        "cliente_id": cliente_id,
        "tipo": tipo,
        "valor": valor,
        "descricao": (linha.get("descricao") or "").strip(),
        "categoria": (linha.get("categoria") or "").strip(),
    }


def limpar_transacoes(brutas):
    """Aplica a validacao a todas as linhas e imprime o resumo da limpeza."""
    validas = []
    for linha in brutas:
        registro = validar_transacao(linha)
        if registro is not None:
            validas.append(registro)

    total = len(brutas)
    n_validas = len(validas)
    print("===== RESUMO DA LIMPEZA =====")
    print(f"Total de linhas lidas: {total}")
    print(f"Linhas validas: {n_validas}")
    print(f"Linhas invalidas: {total - n_validas}")
    return validas

## Agrupamento mensal e metricas

`gerar_relatorio` percorre as transacoes com um dicionario indexado por mes
(`AAAA-MM`) e calcula quantidade, totais de credito/debito, saldo, media,
maior e menor valor. `calcular_periodo` mede os dias entre a transacao mais
antiga e a mais recente.

In [6]:
def gerar_relatorio(transacoes):
    """Agrupa por mes e calcula as metricas mensais."""
    resumo = {}
    for t in transacoes:
        mes = t["mes"]
        if mes not in resumo:
            resumo[mes] = {
                "quantidade": 0,
                "total_credito": 0.0,
                "total_debito": 0.0,
                "saldo": 0.0,
                "media": 0.0,
                "maior_valor": t["valor"],
                "menor_valor": t["valor"],
                "_soma_valores": 0.0,
            }
        m = resumo[mes]
        m["quantidade"] += 1
        m["_soma_valores"] += t["valor"]
        if t["tipo"] == "credito":
            m["total_credito"] += t["valor"]
        else:
            m["total_debito"] += t["valor"]
        m["maior_valor"] = max(m["maior_valor"], t["valor"])
        m["menor_valor"] = min(m["menor_valor"], t["valor"])

    for m in resumo.values():
        m["saldo"] = round(m["total_credito"] - m["total_debito"], 2)
        m["media"] = round(m["_soma_valores"] / m["quantidade"], 2)
        m["total_credito"] = round(m["total_credito"], 2)
        m["total_debito"] = round(m["total_debito"], 2)
        del m["_soma_valores"]

    return dict(sorted(resumo.items()))


def detectar_suspeitas(transacoes):
    """Retorna as transacoes com valor acima do LIMITE_SUSPEITO."""
    return [t for t in transacoes if t["valor"] > LIMITE_SUSPEITO]


def calcular_periodo(transacoes):
    """Retorna (data_min, data_max, dias) do conjunto de transacoes."""
    datas = [t["data"] for t in transacoes]
    data_min, data_max = min(datas), max(datas)
    return data_min, data_max, (data_max - data_min).days

## Exportacao em JSON

In [7]:
def salvar_json(dados, caminho):
    """Salva o relatorio em JSON formatado e com acentos preservados."""
    with open(caminho, "w", encoding="utf-8") as arquivo:
        json.dump(dados, arquivo, ensure_ascii=False, indent=2)
    print(f"Relatorio salvo em '{caminho}'.")

## Exibicao formatada no terminal

In [8]:
def exibir_relatorio(resumo, suspeitas, periodo, n_validas, n_invalidas):
    data_min, data_max, dias = periodo

    print()
    print("===== RELATORIO MENSAL =====")
    for mes, m in resumo.items():
        print()
        print(f"Mes: {mes}")
        print(f"Transacoes: {m['quantidade']}")
        print(f"Total credito: {formatar_brl(m['total_credito'])}")
        print(f"Total debito: {formatar_brl(m['total_debito'])}")
        print(f"Saldo: {formatar_brl(m['saldo'])}")
        print(f"Media: {formatar_brl(m['media'])}")
        print(f"Maior valor: {formatar_brl(m['maior_valor'])}")
        print(f"Menor valor: {formatar_brl(m['menor_valor'])}")

    print()
    print("===== TRANSACOES SUSPEITAS =====")
    if suspeitas:
        for t in suspeitas:
            print(
                f"ID: {t['id']} | Cliente: {t['cliente_id']} | "
                f"Data: {t['data'].strftime('%Y-%m-%d')} | Valor: {formatar_brl(t['valor'])}"
            )
    else:
        print("Nenhuma transacao suspeita encontrada.")

    print()
    print("===== PERIODO ANALISADO =====")
    print(f"De {data_min.strftime('%Y-%m-%d')} ate {data_max.strftime('%Y-%m-%d')} ({dias} dias)")
    print(f"Transacoes validas: {n_validas} | Transacoes invalidas: {n_invalidas}")

## Celula de Execucao Principal

Roda do inicio ao fim: le, limpa, calcula, exibe e salva o JSON.

In [9]:
def main():
    brutas = ler_transacoes(ARQUIVO_CSV)
    if not brutas:
        return

    validas = limpar_transacoes(brutas)
    n_invalidas = len(brutas) - len(validas)

    resumo = gerar_relatorio(validas)
    suspeitas = detectar_suspeitas(validas)
    periodo = calcular_periodo(validas)

    exibir_relatorio(resumo, suspeitas, periodo, len(validas), n_invalidas)

    relatorio = {
        "gerado_em": datetime.now().strftime("%Y-%m-%d"),
        "periodo": {
            "data_inicial": periodo[0].strftime("%Y-%m-%d"),
            "data_final": periodo[1].strftime("%Y-%m-%d"),
            "dias": periodo[2],
        },
        "total_transacoes_validas": len(validas),
        "total_transacoes_invalidas": n_invalidas,
        "resumo_mensal": resumo,
        "transacoes_suspeitas": [
            {
                "id": t["id"],
                "cliente_id": t["cliente_id"],
                "data": t["data"].strftime("%Y-%m-%d"),
                "valor": t["valor"],
            }
            for t in suspeitas
        ],
    }
    print()
    salvar_json(relatorio, ARQUIVO_JSON)


main()

===== RESUMO DA LIMPEZA =====
Total de linhas lidas: 22
Linhas validas: 17
Linhas invalidas: 5

===== RELATORIO MENSAL =====

Mes: 2026-01
Transacoes: 4
Total credito: R$ 15.500,00
Total debito: R$ 600,50
Saldo: R$ 14.899,50
Media: R$ 4.025,12
Maior valor: R$ 12.000,00
Menor valor: R$ 180,50

Mes: 2026-02
Transacoes: 5
Total credito: R$ 18.500,00
Total debito: R$ 609,90
Saldo: R$ 17.890,10
Media: R$ 3.821,98
Maior valor: R$ 15.000,00
Menor valor: R$ 89,90

Mes: 2026-03
Transacoes: 5
Total credito: R$ 3.750,00
Total debito: R$ 2.240,20
Saldo: R$ 1.509,80
Media: R$ 1.198,04
Maior valor: R$ 3.500,00
Menor valor: R$ 99,90

Mes: 2026-04
Transacoes: 3
Total credito: R$ 4.200,00
Total debito: R$ 385,00
Saldo: R$ 3.815,00
Media: R$ 1.528,33
Maior valor: R$ 4.200,00
Menor valor: R$ 75,00

===== TRANSACOES SUSPEITAS =====
ID: 4 | Cliente: CLI003 | Data: 2026-01-25 | Valor: R$ 12.000,00
ID: 6 | Cliente: CLI003 | Data: 2026-02-14 | Valor: R$ 15.000,00

===== PERIODO ANALISADO =====
De 2026-01-05 a